In [1]:
import pandas as pd

CSV_PATH = "revenues_per_day.csv"
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig", dtype="string", keep_default_na=False)
print(f"Rows: {len(df)}")

Rows: 337818


In [2]:
df.isna().sum().rename("missing_values").to_frame()

,missing_values
id,0
date,0
title,0
revenue,0
theaters,161
distributor,225


In [3]:
summary = []
for column in ["distributor", "theaters"]:
    empty = raw[column].eq("")
    whitespace = raw[column].str.strip().eq("") & ~empty
    interpreted_as_na = df[column].isna() & ~(empty | whitespace)
    summary.append({
        "column": column,
        "pandas_missing": int(df[column].isna().sum()),
        "empty_fields": int(empty.sum()),
        "whitespace_only": int(whitespace.sum()),
        "non_empty_na_markers": int(interpreted_as_na.sum()),
    })
    print(f"{column}: original values behind pandas missing values")
    print({repr(value): count for value, count in raw.loc[df[column].isna(), column].value_counts().items()})

pd.DataFrame(summary).set_index("column")

distributor: original values behind pandas missing values
{"'N/A'": np.int64(225)}
theaters: original values behind pandas missing values
{"''": np.int64(161)}


,pandas_missing,empty_fields,whitespace_only,non_empty_na_markers
column,,,,
distributor,225,0,0,225
theaters,161,161,0,0


In [4]:
empty_theaters = df.loc[
    raw["theaters"].str.strip().eq(""),
    ["id", "date", "title", "revenue", "theaters", "distributor"],
].copy()

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(empty_theaters)

,id,date,title,revenue,theaters,distributor
136,055a8612-0597-0769-e33c-92689b575033,2000-02-03,Stuart Little,201662,NaN,Sony Pictures Entertainment (SPE)
1296,177895ed-69d4-f671-1c08-07b6a0268556,2015-05-31,On Tour with Pina Bausch,133,NaN,Icarus Films
5850,09d5c32c-7540-5c5f-2377-52a315ce59cb,2000-02-01,Stuart Little,221984,NaN,Sony Pictures Entertainment (SPE)
9672,99a300ed-028f-683f-af82-bdcd86ca0c75,2017-12-07,The Polar Express2017 IMAX Release,19539,NaN,Warner Bros.
13117,13a2f5d9-da94-19d1-46e6-db141c6325d8,2017-12-21,The Polar Express2017 IMAX Release,13744,NaN,Warner Bros.
13206,cc4d13aa-bede-78e6-e7e5-0960893d50d3,2011-11-26,We Bought a Zoo,458486,NaN,Twentieth Century Fox
31515,e2840f63-18c8-bf06-44f1-1c01437107ad,2016-03-29,I Don't Belong Anywhere: The Cinema of Chantal Akerman,401,NaN,Icarus Films
36627,5573168a-71f8-02e3-d20f-454aa0706977,2017-12-28,The Polar Express2017 IMAX Release,8220,NaN,Warner Bros.
37066,99def74d-e8ad-6371-6278-850509c5411f,2017-12-19,The Polar Express2017 IMAX Release,9900,NaN,Warner Bros.
44893,c2e9e4eb-89d6-f76e-927b-3f265e14f550,2017-12-04,The Polar Express2017 IMAX Release,5437,NaN,Warner Bros.


In [5]:
revenue = empty_theaters["revenue"]
print(f"Rows with empty theaters: {len(empty_theaters)}")
print(f"Missing revenue: {revenue.isna().sum()}")
print(f"Zero revenue: {revenue.eq(0).sum()}")
print(f"Negative revenue: {revenue.lt(0).sum()}")
print(f"Positive revenue: {revenue.gt(0).sum()}")
print(f"Total revenue: {revenue.sum():,.0f}")
print(f"Minimum revenue: {revenue.min():,.0f}")
print(f"Maximum revenue: {revenue.max():,.0f}")
print(f"Median revenue: {revenue.median():,.2f}")

Rows with empty theaters: 161
Missing revenue: 0
Zero revenue: 2
Negative revenue: 0
Positive revenue: 159
Total revenue: 78,310,538
Minimum revenue: 0
Maximum revenue: 39,500,000
Median revenue: 19,673.00


In [6]:
print(f"Missing IDs: {df['id'].isna().sum()}")
print(f"Unique non-null ID for every row: {df['id'].is_unique and df['id'].notna().all()}")
print(f"Duplicate IDs beyond first occurrence: {df['id'].dropna().duplicated().sum()}")

Missing IDs: 0


Unique non-null ID for every row: True
Duplicate IDs beyond first occurrence: 0


In [7]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise")
first_dates = df.groupby("title", as_index=False)["date"].min()
first_dates["first_year"] = first_dates["date"].dt.year
first_year_pairs = first_dates[["title", "first_year"]].drop_duplicates()

print(f"Rows: {len(df)}")
print(f"Unique titles: {df['title'].nunique()}")
print(f"Unique title-first year pairs: {len(first_year_pairs)}")

Rows: 337818
Unique titles: 6545
Unique title-first year pairs: 6545
